In [1]:
import pandas as pd
import numpy as np
import scikit_na as na
import missdat

In [2]:
def generate_MCAR(X, missing_column_name, missing_fraction=0.10, random_state=42):
    df_temp = X.copy()
    np.random.seed(random_state)
    n_count = len(df_temp)
    if not n_count:
        raise ValueError('Empty DataFrame')
    n_missing = int(missing_fraction * n_count)
    missing_index = np.random.choice(n_count, n_missing, replace=False)
    df_temp.loc[missing_index, missing_column_name] = np.nan
    return df_temp

In [3]:
def generate_MAR_MNAR(X, missing_column_name, condition_column, missing_fraction=0.10, random_state=42):
    df_temp = X.copy()
    np.random.seed(random_state)
    condition_index = df_temp[df_temp[condition_column] == True].index
    n_count = len(condition_index)
    if not n_count:
        raise ValueError('No row satisfies the condition to be filled with NaN')
    n_missing = int(missing_fraction * n_count)
    missing_index = np.random.choice(condition_index, n_missing, replace=False)
    df_temp.loc[missing_index, missing_column_name] = np.nan
    df_temp.drop(columns=condition_column, inplace=True)
    return df_temp

In [4]:
n_samples = 5000
X = pd.DataFrame({
    'A' : np.random.randn(n_samples),
    'B' : np.random.randn(n_samples),
    'C' : np.random.randn(n_samples)
})
X.head()

,A,B,C
0,-0.415907,-0.556314,-0.574663
1,-0.357844,-2.176414,0.474865
2,-0.904895,-0.534227,1.738587
3,1.273981,-0.077626,3.110778
4,-0.105766,-0.346815,0.715216


### Missing Completely at Random
The probability of an attribute missing does not depend on the observed data but is due to random unknown factors.

In [5]:
random_state = 42
MCAR = X.copy()

for col in MCAR.columns:
    MCAR = generate_MCAR(MCAR, col, random_state=random_state)
    random_state +=  1

MCAR.isna().sum().sum()

np.int64(1500)

In [6]:
summary = na.summary(MCAR)
display(summary)

,A,B,C
na_count,500.00,500.00,500.00
na_pct_per_col,10.00,10.00,10.00
na_pct_total,33.33,33.33,33.33
na_unique_per_col,408.00,400.00,401.00
na_unique_pct_per_col,81.60,80.00,80.20
rows_after_dropna,4500.00,4500.00,4500.00
rows_after_dropna_pct,90.00,90.00,90.00


In [7]:
na.altair.plot_heatmap(MCAR).properties(width=200, height=400)

alt.Chart(...)

In [8]:
miss, cor, all, pat, item = missdat.miss_diagnostics(MCAR)
display(miss)

,A,B,C
0,0,0,0
1,0,0,0
2,0,0,0
3,0,0,0
4,0,0,0
...,...,...,...
4995,1,0,0
4996,1,0,0
4997,0,0,0
4998,0,0,0


In [9]:
display(pat)

,A,B,C,n
Missing Pattern 1,0,0,0,3647
Missing Pattern 2,0,0,1,401
Missing Pattern 3,0,1,0,400
Missing Pattern 4,0,1,1,52
Missing Pattern 5,1,0,0,408
Missing Pattern 6,1,0,1,44
Missing Pattern 7,1,1,0,45
Missing Pattern 8,1,1,1,3


In [10]:
display(cor)

,A,B,C
A,1.000000,-0.004444,-0.006667
B,-0.004444,1.000000,0.011111
C,-0.006667,0.011111,1.000000


In [11]:
display(all)

,overall missing stats
missing_patterns,8.0
proportion_missing,0.1
proportion_complete,0.9


In [12]:
display(item)

,number_missing,proportion_missing,proportion_complete
A,500,0.1,0.9
B,500,0.1,0.9
C,500,0.1,0.9


In [13]:
missdat.mcar_test(MCAR)

,MCAR Test Values
number of missing patterns,8
x2,6.780136
df,18.0
p,0.991853
alpha,0.05
interpretation,Missing Completely at Random (MCAR)


In [14]:
# MCAR data, coefficients will not be significant
subset = MCAR.loc[:, MCAR.dtypes != object]
model = na.model(subset, col_na='B')
model.summary()

Optimization terminated successfully.
         Current function value: 0.322320
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                      B   No. Observations:                 4047
Model:                          Logit   Df Residuals:                     4044
Method:                           MLE   Df Model:                            2
Date:                Fri, 05 Dec 2025   Pseudo R-squ.:               0.0006307
Time:                        23:24:45   Log-Likelihood:                -1304.4
converged:                       True   LL-Null:                       -1305.3
Covariance Type:            nonrobust   LLR p-value:                    0.4390
===============================================================================
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
(intercept)    -2.2126      0.053    -41.916      0.000      -2.316      -2.109
A              -0.0370      0.053     -0.699      0.484      -0.141       0.067
C               0.0564      0.052      1.083      0.279      -0.046       0.159
===============================================================================
"""

### Missing at Random
The probability of an attribute missing depends on the the observed data.

In [15]:
MAR = X.copy()
MAR['MAR_condition_column'] = (3 * MAR['A'] + MAR['C'] > 0)
MAR = generate_MAR_MNAR(MAR, 'B', 'MAR_condition_column')
MAR.isna().sum().sum()

np.int64(247)

In [16]:
miss, cor, all, pat, item = missdat.miss_diagnostics(MAR)

In [17]:
display(pat)

,A,B,C,n
Missing Pattern 1,0,0,0,4753
Missing Pattern 2,0,1,0,247


In [18]:
missdat.mcar_test(MAR)

,MCAR Test Values
number of missing patterns,2
x2,207.432179
df,9.0
p,0.0
alpha,0.05
interpretation,**Not** Random (Potentially MAR or MNAR)


In [19]:
# MAR data, coefficients will be significant
subset = MAR.loc[:, MAR.dtypes != object]
model = na.model(subset, col_na='B')
model.summary()

Optimization terminated successfully.
         Current function value: 0.175450
         Iterations 8


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                      B   No. Observations:                 5000
Model:                          Logit   Df Residuals:                     4997
Method:                           MLE   Df Model:                            2
Date:                Fri, 05 Dec 2025   Pseudo R-squ.:                  0.1082
Time:                        23:24:48   Log-Likelihood:                -877.25
converged:                       True   LL-Null:                       -983.72
Covariance Type:            nonrobust   LLR p-value:                 5.733e-47
===============================================================================
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
(intercept)    -3.3847      0.089    -37.933      0.000      -3.560      -3.210
A               0.9461      0.072     13.228      0.000       0.806       1.086
C               0.2980      0.067      4.477      0.000       0.168       0.428
===============================================================================
"""

### Missing Not at Random
The probability of an attribute missing depends on the value of the missing attribute itself. Detected as MCAR by Little's Test.

In [20]:
MNAR = X.copy()
MNAR['MNAR_condition_column'] = (MNAR['B'] < 0.5) & (MNAR['B'] > 0.1)
MNAR = generate_MAR_MNAR(MNAR, 'B', 'MNAR_condition_column')
MNAR.isna().sum().sum()

np.int64(76)

In [21]:
miss, cor, all, pat, item = missdat.miss_diagnostics(MNAR)
display(pat)

,A,B,C,n
Missing Pattern 1,0,0,0,4924
Missing Pattern 2,0,1,0,76


In [22]:
# MNAR will be detected as MCAR as the data missingness is due to 'unknown' factors (the value of the missing attribute)
missdat.mcar_test(MNAR)

,MCAR Test Values
number of missing patterns,2
x2,0.239844
df,9.0
p,0.999999
alpha,0.05
interpretation,Missing Completely at Random (MCAR)


In [23]:
# MNAR data, coefficients will not be significant
subset = MNAR.loc[:, MNAR.dtypes != object]
model = na.model(subset, col_na='B')
model.summary()

Optimization terminated successfully.
         Current function value: 0.078694
         Iterations 8


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                      B   No. Observations:                 5000
Model:                          Logit   Df Residuals:                     4997
Method:                           MLE   Df Model:                            2
Date:                Fri, 05 Dec 2025   Pseudo R-squ.:               0.0003048
Time:                        23:24:50   Log-Likelihood:                -393.47
converged:                       True   LL-Null:                       -393.59
Covariance Type:            nonrobust   LLR p-value:                    0.8870
===============================================================================
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
(intercept)    -4.1727      0.116    -36.043      0.000      -4.400      -3.946
A               0.0041      0.116      0.036      0.971      -0.222       0.231
C              -0.0556      0.114     -0.489      0.625      -0.278       0.167
===============================================================================
"""